# C1-ml-fundamentals — Practice p14 — Solution

**Challenge.** A sensor produces one reading per machine part; faulty parts
(class `1`, about 1 in 5) tend to read higher. Instead of picking a single
cutoff, you will sweep *every* candidate cutoff and watch precision and
recall trade off against each other.

The data cell below generates 200 labeled readings and splits them
150 train / 50 test.

**(a)** Build `thresholds`: the midpoints between consecutive *sorted*
training readings — shape (149,).

**(b)** Compute three arrays of shape (149,): `precisions`, `recalls`, and
`f1s`, where entry *i* scores the rule "flag a part when its reading is at
least `thresholds[i]`" **on the training set**. **Banned in this part:
Python `for` and `while` loops and list comprehensions — any use scores zero
points.** Hint: broadcasting (as covered in F1-scientific-python) —
`X_train[None, :] >= thresholds[:, None]` is a (149, 150) table of
predictions, and axis-wise sums finish the job. Some cutoffs flag nothing;
use `np.where` so those entries get precision 0 and F1 0 instead of a
division by zero.

**(c)** Find `best_t`, the threshold with the highest **training** F1, then
compute `test_precision`, `test_recall`, and `test_f1` for that single
cutoff on the held-out test set (exact names).

**(d)** Plot `precisions` and `recalls` against `thresholds` as two labeled
lines with a legend, axis labels, and a vertical line at `best_t`. Describe
in one markdown sentence what the two curves do as the threshold rises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
SEED = 20260804
data_rng = np.random.default_rng(SEED)
readings_ok    = data_rng.normal(2.5, 1.0, size=160)   # healthy parts
readings_fault = data_rng.normal(5.0, 1.2, size=40)    # faulty parts

X = np.concatenate([readings_ok, readings_fault])
y = np.concatenate([np.zeros(160, dtype=int), np.ones(40, dtype=int)])

order = data_rng.permutation(200)
X, y = X[order], y[order]
X_train, y_train = X[:150], y[:150]
X_test,  y_test  = X[150:], y[150:]

In [ ]:
# (a) candidate cutoffs: midpoints of the sorted training readings
s = np.sort(X_train)
thresholds = (s[:-1] + s[1:]) / 2
print(thresholds.shape)

# (b) the whole sweep at once, via broadcasting
preds = X_train[None, :] >= thresholds[:, None]        # (149, 150) bool
TPs = np.sum(preds & (y_train == 1)[None, :], axis=1)  # per-threshold counts
FPs = np.sum(preds & (y_train == 0)[None, :], axis=1)
FNs = np.sum(~preds & (y_train == 1)[None, :], axis=1)

flagged = TPs + FPs
precisions = np.where(flagged > 0, TPs / np.maximum(flagged, 1), 0.0)
recalls    = TPs / (TPs + FNs)          # denominator is always the 30+ positives
pr_sum     = precisions + recalls
f1s = np.where(pr_sum > 0, 2 * precisions * recalls / np.maximum(pr_sum, 1e-12), 0.0)
print(precisions.shape, recalls.shape, f1s.shape)

In [ ]:
# (c) best training-F1 cutoff, then the honest test-set measurement
best_i = int(np.argmax(f1s))
best_t = float(thresholds[best_i])
print(f"best_t = {best_t:.3f}  (training F1 = {f1s[best_i]:.3f})")

test_flag = X_test >= best_t
TP = np.sum(test_flag & (y_test == 1))
FP = np.sum(test_flag & (y_test == 0))
FN = np.sum(~test_flag & (y_test == 1))
test_precision = TP / (TP + FP)
test_recall    = TP / (TP + FN)
test_f1        = 2 * test_precision * test_recall / (test_precision + test_recall)
print(f"test precision = {test_precision:.3f}  recall = {test_recall:.3f}  F1 = {test_f1:.3f}")

In [ ]:
# (d) the precision-recall tradeoff picture
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, precisions, color="#2a78d6", linewidth=2, label="precision")
ax.plot(thresholds, recalls,    color="#eb6834", linewidth=2, label="recall")
ax.axvline(best_t, color="#555555", linewidth=2, linestyle="--",
           label=f"best F1 cutoff = {best_t:.2f}")
ax.set_xlabel("threshold (sensor reading)")
ax.set_ylabel("score on training set")
ax.set_title("Precision and recall trade off as the cutoff rises")
ax.legend(loc="center left")
plt.show()

**(d) description.** As the threshold rises, recall falls (ever more faulty
parts sit below the cutoff and are missed) while precision generally climbs
toward 1 (the few parts still flagged are the extreme readings, almost all
truly faulty) — the two curves pull in opposite directions, and the best-F1
cutoff ≈ 4.12 sits where the two are jointly strong.
**Explanation of the whole problem:** the broadcast comparison scores all 149
candidate rules in one shot with no loop, the `np.where` guards keep
empty-flag cutoffs at precision 0 rather than 0/0, and the final numbers are
measured on the untouched test set — the cutoff was *chosen* on training data
only, keeping the train/test discipline intact.

### Answer check

In [ ]:
assert thresholds.shape == (149,)
assert precisions.shape == (149,) and recalls.shape == (149,) and f1s.shape == (149,)
assert np.isclose(best_t, 4.120487934911861, atol=1e-9, rtol=0)
assert np.isclose(f1s[best_i], 0.8813559322033899, atol=1e-9, rtol=0)
assert np.isclose(test_precision, 1.0, atol=1e-9, rtol=0)
assert np.isclose(test_recall, 0.5555555555555556, atol=1e-9, rtol=0)
assert np.isclose(test_f1, 0.7142857142857143, atol=1e-9, rtol=0)
# recall is non-increasing as the cutoff rises
assert np.all(np.diff(recalls) <= 1e-12)
print("p14 OK")